#### Handling sequences with PyTorch

Sequential data
- Ordered in time or space
- Order of the data points contains dependencies between them
- Examples of sequential data:
    - Time series
    - Text
    - Audio waves

In [1]:
import pandas as pd
import torch
import matplotlib.pyplot as mlt
import numpy as np


ELectricity consumption prediction
- Task: predict future electricity consumption based on past patterns
- Electricity consumption dataset:

In [2]:
df = pd.read_csv('electricity_consump/electricity_train.csv')
df.head()

,timestamp,consumption
0,2011-01-01 00:15:00,-0.704319
1,2011-01-01 00:30:00,-0.704319
2,2011-01-01 00:45:00,-0.678983
3,2011-01-01 01:00:00,-0.653647
4,2011-01-01 01:15:00,-0.704319


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105215 entries, 0 to 105214
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   timestamp    105215 non-null  object 
 1   consumption  105215 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.6+ MB


In [4]:
df.describe()

,consumption
count,105215.000000
mean,-0.007469
std,1.056835
min,-1.414483
25%,-0.957931
50%,-0.349363
75%,0.797593
max,3.028924


Train-test split
- No random splitting for time series!
- Look-ahead bias: model has info about the future
- Solution: split by time

Creating Sequences
- Sequence length = number of data points in one training example
    - 24 x 4 = 96 -> consider last 24 hours
- Predict single next data point

Creating sequences in Python
- Take data and sequence length as inputs
- Initialize imputs and targets lists
- Iterate over data points
- Define inputs and target
- Append to pre-initialized lists
- Return inputs and targets as NumPy arrays

In [5]:
import numpy as np

def create_sequences(df, seq_length):
    xs, ys = [], []
    for i in range(len(df) - seq_length):
        x = df.iloc[i: (i+seq_length), 1]
        y = df.iloc[i+seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

### TensorDataset
- Create training examples

In [6]:
seq_length = 24*4
X_train, y_train = create_sequences(df, seq_length)
print(X_train.shape,  y_train.shape)

(105119, 96) (105119,)


Convert them to a Torch Dataset

In [7]:
from torch.utils.data import TensorDataset

dataset_train = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)

#### Applicability to other sequential data
Same techniques are applicable to other sequences:
- Large Language Models
- Speech recognition

Exercise:
- Generating sequences

In [8]:
import numpy as np

def create_sequences(df, seq_length):
    xs, ys = [], []
    # Iterate over data indices
    for i in range(len(df) - seq_length):
      	# Define inputs
        x = df.iloc[i:(i + seq_length), 1]
        # Define target
        y = df.iloc[i+seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

- Sequential Dataset

In [9]:
import torch
from torch.utils.data import TensorDataset
train_data = df
# Use create_sequences to create inputs and targets
X_train, y_train = create_sequences(train_data, seq_length=24*4)
print(X_train.shape, y_train.shape)

# Create TensorDataset
dataset_train = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)
print(len(dataset_train))

(105119, 96) (105119,)
105119


### Recurrent Neural Networks
Recurrent neuron
- Feed-forward networks
- RNNs: have connections pointing back
- Recurrent neuron:
    - Input `x`
    - Output `y`
    - Hidden state `h`
- In PyTorch: `nn.RNN()`

### Sequence-to-sequence architecture
- Pass sequence as input, use the entire output sequence
- Example: Real-time speech recognition
### Sequence-to-vector architecture
- Pass sequence as input, use only the last output
- Example: Text topic classification
### Vector-to-sequence architecture
- Pass single input, use the entire output sequence
- Example: Text generation
### Encoder-decoder architecture
- Pass entire input sequence, only then start using output sequence
- Example: Machine translation

### RNN In PyTorch
- Sequence-to-vector
    - Define model class with `__init__` method
    - Define recurrent layer. `self.rnn`
    - Define linear layers, `fc`
    - In `forward()`, initialize first hidden state to zeros
    - Pass input and first hidden state throuh RNN layer
    - Select last RNN's output and pass it through linear layer



In [10]:
import torch
import torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

Exercise:
- Building a forecasting RNN

In [11]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        # Initialize first hidden state with zeros
        h0 = torch.zeros(2, x.size(0), 32)
        # Pass x and h0 through recurrent layer
        out, _ = self.rnn(x, h0)  
        # Pass recurrent layer's last output through linear layer
        out = self.fc(out[:, -1, :])
        return out

### LSTM and GRU cells

### SHort-term memory problem
- RNN cells maintain memory via hidden state
- This memory is very short-term
- Two more powerful cells solve the problem:
    - LSTM (Long Short-Term Memory) cell
    - GRU (Gsted Recurrent Unit) cell

#### RNN cell:
- Two Inputs:
    - Current input data `x`
    - Previous hidden state `h`

#### LSTM cell:
- Three inputs and outputs (two hideen states):
    - `h`: short-term state (memory)
    - `c`: long-term state (memory)
- Three "gates":
    - `Forget gate`: what to remove from long-term memory
    - `Input gate`: what to save to long-term memory
    - `Output gate`: waht to return at the current time step


#### LSTM in PyTorch
- `__init__()`:
    - Replace `nn.RNN` with `nn.LSTM`
- `forward()`:
    - Add another hidden state `c`
    - Initialize `c` and `h` with zeros
    - Pass both hidden states to `lstm` layer

In [12]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True, 
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        c0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

### GRU cell
- Simplified version of LSTM cell
- Just one hidden state
- No output gate



#### GRU in PyTorch
- `__init__()`:
    - Replace `nn.RNN` with `nn.GRU`
- `forward()`:
    - Use the `gru` layer

In [13]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

### Should I use RNN, LSTM, or GRU?
- RNN is not used much  (because of short term memory problem)
- GRU is simple than LSTM = less computation
- Relative performance varies per use-case
- Try both and compare

Exercise:
- LSTM network

In [14]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        # Define lstm layer
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        # Initialize long-term memory
        c0 = torch.zeros(2, x.size(0), 32)
        # Pass all inputs to lstm layer
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

- GRU network

In [15]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.gru(x, h0)  
        out = self.fc(out[:, -1, :])
        return out

### Training and evaluating RNNs
Mean Squared Error Loss
- Error: prediction - target
- Squared Error: (prediction - target)^2
- Mean Squared Error: avg[(prediction - target)^2]

Squaring the error:
- Ensures positive and negative errors dont cancel out
- penalizes large errors more
- PyTorch: `criterion = nn.MSELoss()`

Expanding tensors
- Recurrent layers expects input shape
`(batch_size, seq_length, num_features)`
- We got `(batch_size, seq_length)`
- we must add one dimension at the end

In [16]:
from torch.utils.data import DataLoader

dataloader_train = DataLoader(
    dataset_train,
    batch_size=32,
    shuffle=True,
)

for seqs, labels in dataloader_train:
    print(seqs.shape)
    break

torch.Size([32, 96])


In [17]:
seqs = seqs.view(32, 96, 1)
print(seqs.shape)

torch.Size([32, 96, 1])


Squeezing tensors
- In evaluation loop, we need to revert the reshaping done in the training loop
- Labels are of shape `(batch_size)`

In [18]:
df_test = pd.read_csv('electricity_consump/electricity_test.csv')
X_test, y_test = create_sequences(df_test, seq_length=24*4)
dataset_test = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).float(),
)

test_loader = DataLoader(
    dataset_test,
    batch_size=32,
    shuffle=False,
)

for seqs, labels in test_loader:
    print(labels.shape)
    break

torch.Size([32])


- Model outputs are `(batch_size, 1)`

In [19]:
net = Net()
seqs, labels = next(iter(dataloader_train))
seqs = seqs.view(seqs.size(0), seqs.size(1), 1)
out = net(seqs)
print(out.shape)

torch.Size([32, 1])


- Shapes of models outputs and labels must match for the loss function
- we can drop the last dimension from model outputs

In [20]:
out = net(seqs).squeeze()
print(out.shape)

torch.Size([32])


### Training Loop
- Instantiate model, define loss & optimizer
- iterate over epochs and data batches
- Reshape input sequence
- The rest: as usual

In [24]:
import torch.optim as optim

num_epochs = 3
net = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(
    net.parameters(), lr=0.001
)

for epoch in range(num_epochs):
    for seqs, labels in dataloader_train:
        seqs = seqs.view(32, 96, 1)
        outputs = net(seqs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

RuntimeError: shape '[32, 96, 1]' is invalid for input of size 2976

### Evaluation loop
- Set up MSE metric
- Iterate through test data with no gradients
- Reshape model inputs
- Squeeze model outputs
- Update the metric
- Compute final metric value

In [ ]:
import torchmetrics
mse = torchmetrics.MeanSquaredError()

net.eval()
with torch.no_grad():
    for seqs, labels in test_loader:
        seqs = seqs.view(32, 96, 1)
        outputs = net(seqs).squeeze()
        mse(outputs, labels)

print(f"Test MSE: {mse.compute()}")
    

### Exercise:
- RNN training loop

In [ ]:
net = Net()
# Set up MSE loss
criterion = nn.MSELoss()
optimizer = optim.Adam(
  net.parameters(), lr=0.0001
)

for epoch in range(3):
    for seqs, labels in dataloader_train:
        # Reshape model inputs
        seqs = seqs.view(32, 96, 1)
        # Get model outputs
        outputs = net(seqs)
        # Compute loss
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

- Evaluating forecasting models

In [ ]:
# Define MSE metric
mse = torchmetrics.MeanSquaredError()

net.eval()
with torch.no_grad():
    for seqs, labels in dataloader_test:
        seqs = seqs.view(32, 96, 1)
        # Pass seqs to net and squeeze the result
        outputs = net(seqs).squeeze()
        mse(outputs, labels)

# Compute final metric value
test_mse = mse.compute()
print(f"Test MSE: {test_mse}")